# Question 4: ETL on web-scraped quotes, then SQL analysis

**Analytics flow**
1. **4.1 Extract**: scrape two pages of <https://quotes.toscrape.com/> (20 quotes) and deliberately introduce dirty values.
2. **4.2 Transform**: handle missing text and authors, parse messy tag counts, derive `word_count`, impute missing tag counts with the median, export a clean CSV.
3. **4.3 Load and analyse**: load the CSV into PostgreSQL (`quote_listings`) and run two SQL investigations.

Running 4.1 and 4.2 **re-creates** `data/q4_quotes/cleaned_quotes_dataset.csv`.

In [ ]:
library(rvest)
library(stringr)
library(dplyr)
library(readr)

## 4.1 Extract

In [ ]:
get_page <- function(u) {
  page  <- read_html(u)
  boxes <- page %>% html_nodes(".quote")

  data.frame(
    quote_text    = page %>% html_nodes(".quote .text")   %>% html_text(),
    author        = page %>% html_nodes(".quote .author") %>% html_text(),
    tag_count_str = as.character(sapply(boxes, function(b) {
      length(html_nodes(b, ".tags .tag"))
    })),
    stringsAsFactors = FALSE
  )
}

df_raw <- bind_rows(
  get_page("https://quotes.toscrape.com/"),
  get_page("https://quotes.toscrape.com/page/2/")
)

nrow(df_raw)   # should be 20

In [ ]:
# Introduce deliberate "dirty" values to demonstrate cleaning
set.seed(42)
df_raw$tag_count_str[c(4, 15)] <- c("approx 3 tags", NA)
df_raw$author[9]               <- ""
df_raw$quote_text[12]          <- NA

head(df_raw, 5)

**Recorded output** (screenshot from the project documentation): raw scraped data with injected dirty values

![raw scraped data with injected dirty values](../docs/images/q4_1_scrape.png)

## 4.2 Transform and export

In [ ]:
df_clean <- df_raw %>%
  mutate(
    quote_text = ifelse(is.na(quote_text) | quote_text == "", "[Missing Quote Content]", quote_text),
    author     = ifelse(is.na(author) | author == "", "Anonymous", author)
  ) %>%
  mutate(
    extracted_tags = str_extract(tag_count_str, "[0-9]+"),
    numeric_tags   = as.numeric(extracted_tags)
  ) %>%
  mutate(word_count = str_count(quote_text, "\\S+"))

# Impute missing tag counts with the median (rounded so it fits an INT column)
median_tags <- round(median(df_clean$numeric_tags, na.rm = TRUE))

df_clean <- df_clean %>%
  mutate(numeric_tags = as.integer(coalesce(numeric_tags, median_tags))) %>%
  select(quote_text, author, word_count, numeric_tags)

write_csv(df_clean, "../data/q4_quotes/cleaned_quotes_dataset.csv")
print(paste("Dataset cleaned and exported with", nrow(df_clean), "records."))
df_clean

## 4.3 Load into PostgreSQL and analyse with SQL

The schema and both investigations are in `src/q4_webscraping_quotes.sql`:

In [ ]:
cat(readLines("../src/q4_webscraping_quotes.sql"), sep = "\n")

Load the CSV in pgAdmin (Import/Export) or with `psql`:

```sql
\copy quote_listings(quote_text, author, word_count, numeric_tags)
  FROM 'data/q4_quotes/cleaned_quotes_dataset.csv' CSV HEADER
```

**Recorded output** (screenshot from the project documentation): quote_listings table in PostgreSQL

![quote_listings table in PostgreSQL](../docs/images/q4_3_postgres_table.png)

The same two investigations, reproduced with `dplyr` so the analysis can be followed without a database.

### Investigation 1: author productivity and verbosity

In [ ]:
inv1 <- df_clean %>%
  group_by(author) %>%
  summarise(total_quotes   = n(),
            avg_word_count = round(mean(word_count), 1),
            total_tags     = sum(numeric_tags),
            .groups = "drop") %>%
  filter(total_quotes > 1) %>%
  arrange(desc(total_quotes), desc(avg_word_count))

inv1

**Recorded output** (screenshot from the project documentation): SQL investigation 1 result

![SQL investigation 1 result](../docs/images/q4_3_sql_investigation1.png)

### Investigation 2: categorisation by complexity
A quote is *Detailed* when its word count is above the overall average, otherwise *Concise*.

In [ ]:
avg_wc <- mean(df_clean$word_count)

inv2 <- df_clean %>%
  mutate(complexity_tier = if_else(word_count > avg_wc, "Detailed Quote", "Concise Quote")) %>%
  group_by(complexity_tier) %>%
  summarise(total_quotes   = n(),
            avg_word_count = round(mean(word_count), 1),
            shortest       = min(word_count),
            longest        = max(word_count),
            avg_tags       = round(mean(numeric_tags), 1),
            .groups = "drop") %>%
  arrange(desc(avg_word_count))

inv2

**Recorded output** (screenshot from the project documentation): SQL investigation 2 result

![SQL investigation 2 result](../docs/images/q4_3_sql_investigation2.png)

### Optional: run the SQL from R
Set `RUN_DB <- TRUE` and fill in your connection details to run the investigations against your own PostgreSQL instance.

In [ ]:
RUN_DB <- FALSE

if (isTRUE(RUN_DB)) {
  library(DBI)
  con <- dbConnect(RPostgres::Postgres(), dbname = "WebScarpping_Q4.3",
                   host = "localhost", user = "postgres", password = Sys.getenv("PGPASSWORD"))
  dbGetQuery(con, "SELECT COUNT(*) FROM quote_listings")
  dbDisconnect(con)
}